In [ ]:
# --- Paths (repo-relative; this notebook runs from notebooks/) ---
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
DATA      = PROJECT_ROOT / 'data'
RAW       = DATA / 'raw'          # external source data (read-only)
EXTERNAL  = DATA / 'external'     # frozen third-party / HPC-derived inputs (read-only)
ANNOTATED = DATA / 'annotated'    # tables derived by these notebooks
FIGURES   = PROJECT_ROOT / 'figures'



# Merge TFIso PPI counts into the TF isoform master table
add:
- `on_IsoTF`: whether the exact gene + amino-acid sequence appears in the TFIso library
- `total_PPI`: number of unique positive PPI partners observed across **any tested TFIso isoform of that gene**
- `#_of_PPIs`: number of unique positive PPI partners observed for that specific isoform
- `%_of_PPIs`: `#_of_PPIs / total_PPI`

### Interpretation
`#_of_PPIs` counts positive Y2H partners in this experiment. It is not the complete cellular interactome.

A missing value means the isoform was not successfully tested in the PPI dataset.  
A value of `0` means it had at least one interpretable Y2H test but no positive partners.

In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 150)
pd.set_option("display.width", 180)


## 1. Load the three input tables

In [ ]:

PROJECT_DIR = Path("..")
PPI_DIR = PROJECT_DIR / "data/raw/ppi"

MASTER_PATH = PROJECT_DIR / "data/annotated/TranscriptionIsoforms_df_geneage.csv"
TFISO_LIBRARY_PATH = PPI_DIR / "Table_S1.tsv"
PPI_PATH = PPI_DIR / "SuppTable_PairwiseY2HResults.txt"

OUTPUT_PATH = (
    PROJECT_DIR
    / "data/annotated"
    / "TranscriptionIsoforms_df_geneage_TFIso_PPI.csv"
)
AUDIT_PATH = PROJECT_DIR / "data/annotated/TFIso_sequence_match_audit.csv"

master_df = pd.read_csv(MASTER_PATH)
tfiso_library_df = pd.read_csv(TFISO_LIBRARY_PATH, sep="\t")
ppi_raw_df = pd.read_csv(PPI_PATH, sep="\t")

print("Master table:", master_df.shape)
print("TFIso library:", tfiso_library_df.shape)
print("Pairwise Y2H table:", ppi_raw_df.shape)

display(tfiso_library_df.head())
display(ppi_raw_df.head())


## 2. Identify the master gene and sequence columns

In [ ]:
# Set either variable manually if automatic detection chooses the wrong column.
MASTER_GENE_COL = "base_accession"
MASTER_SEQUENCE_COL = "Sequence"

gene_candidates = [
    "gene_name",
    "gene_symbol",
    "gene",
    "symbol",
    "Gene Names"
]

sequence_candidates = [
    "sequence",
    "aa_seq",
    "protein_sequence",
    "protein_seq",
    "seq"
]

if MASTER_GENE_COL is None:
    MASTER_GENE_COL = next(
        (column for column in gene_candidates if column in master_df.columns),
        None
    )

if MASTER_SEQUENCE_COL is None:
    MASTER_SEQUENCE_COL = next(
        (column for column in sequence_candidates if column in master_df.columns),
        None
    )

if MASTER_GENE_COL is None:
    raise ValueError(
        "Could not identify the gene-symbol column. "
        "Set MASTER_GENE_COL manually."
    )

if MASTER_SEQUENCE_COL is None:
    raise ValueError(
        "Could not identify the amino-acid sequence column. "
        "Set MASTER_SEQUENCE_COL manually."
    )

print("Master gene column:", MASTER_GENE_COL)
print("Master sequence column:", MASTER_SEQUENCE_COL)


## 3. Clean the Y2H calls and calculate PPI counts

In [ ]:

def normalize_gene(series):
    return (
        series.astype("string")
        .str.strip()
        .str.split(r"[;,\s]+", regex=True)
        .str[0]
        .str.upper()
    )


def normalize_sequence(series):
    return (
        series.astype("string")
        .str.replace(r"\s+", "", regex=True)
        .str.replace("*", "", regex=False)
        .str.upper()
    )


def parse_y2h_call(series):
    lookup = {
        "true": True,
        "false": False,
        "1": True,
        "0": False
    }
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .map(lookup)
        .astype("boolean")
    )


required_library_columns = {"clone_id", "gene_symbol", "aa_seq"}
required_ppi_columns = {
    "ad_clone_id", "ad_gene_symbol",
    "db_gene_symbol", "Y2H_result"
}

missing_library = required_library_columns - set(tfiso_library_df.columns)
missing_ppi = required_ppi_columns - set(ppi_raw_df.columns)

if missing_library:
    raise ValueError(
        f"TFIso library is missing columns: {sorted(missing_library)}"
    )

if missing_ppi:
    raise ValueError(
        f"PPI table is missing columns: {sorted(missing_ppi)}"
    )

library = tfiso_library_df.copy()
library["gene_key"] = normalize_gene(library["gene_symbol"])
library["sequence_key"] = normalize_sequence(library["aa_seq"])

ppi = ppi_raw_df.copy()
ppi["Y2H_call"] = parse_y2h_call(ppi["Y2H_result"])

ppi = ppi.merge(
    library[["clone_id", "gene_key", "sequence_key"]],
    left_on="ad_clone_id",
    right_on="clone_id",
    how="left",
    validate="many_to_one"
)

print(
    "Y2H rows missing a matching TFIso sequence:",
    int(ppi["sequence_key"].isna().sum())
)

tested_ppi = ppi[ppi["Y2H_call"].notna()].copy()
positive_ppi = ppi[ppi["Y2H_call"].eq(True)].copy()

isoform_tested_summary = (
    tested_ppi
    .groupby(["gene_key", "sequence_key"], as_index=False)
    .agg(n_tested_partners=("db_gene_symbol", "nunique"))
)

isoform_positive_summary = (
    positive_ppi
    .groupby(["gene_key", "sequence_key"], as_index=False)
    .agg(**{"#_of_PPIs": ("db_gene_symbol", "nunique")})
)

isoform_ppi_summary = isoform_tested_summary.merge(
    isoform_positive_summary,
    on=["gene_key", "sequence_key"],
    how="left",
    validate="one_to_one"
)

isoform_ppi_summary["#_of_PPIs"] = (
    isoform_ppi_summary["#_of_PPIs"]
    .fillna(0)
    .astype(int)
)

genes_with_usable_tests = (
    tested_ppi[["gene_key"]]
    .drop_duplicates()
)

gene_positive_summary = (
    positive_ppi
    .groupby("gene_key", as_index=False)
    .agg(total_PPI=("db_gene_symbol", "nunique"))
)

gene_ppi_summary = genes_with_usable_tests.merge(
    gene_positive_summary,
    on="gene_key",
    how="left",
    validate="one_to_one"
)

gene_ppi_summary["total_PPI"] = (
    gene_ppi_summary["total_PPI"]
    .fillna(0)
    .astype(int)
)

isoform_ppi_summary = isoform_ppi_summary.merge(
    gene_ppi_summary,
    on="gene_key",
    how="left",
    validate="many_to_one"
)

isoform_ppi_summary["%_of_PPIs"] = np.where(
    isoform_ppi_summary["total_PPI"] > 0,
    isoform_ppi_summary["#_of_PPIs"]
    / isoform_ppi_summary["total_PPI"],
    np.nan
)

display(isoform_ppi_summary.head(10))


## 4. Match exact TFIso sequences to the master table and add the four columns

In [ ]:

master_with_keys = master_df.copy()
master_with_keys["gene_key"] = normalize_gene(
    master_with_keys[MASTER_GENE_COL]
)
master_with_keys["sequence_key"] = normalize_sequence(
    master_with_keys[MASTER_SEQUENCE_COL]
)

library_sequence_map = (
    library
    .groupby(["gene_key", "sequence_key"], as_index=False)
    .agg(
        TFIso_clone_ids=(
            "clone_id",
            lambda values: ";".join(sorted(set(values)))
        )
    )
)

match_table = library_sequence_map.merge(
    isoform_ppi_summary[
        [
            "gene_key",
            "sequence_key",
            "total_PPI",
            "#_of_PPIs",
            "%_of_PPIs"
        ]
    ],
    on=["gene_key", "sequence_key"],
    how="left",
    validate="one_to_one"
)

merged_master_df = master_with_keys.merge(
    match_table,
    on=["gene_key", "sequence_key"],
    how="left",
    validate="many_to_one"
)

merged_master_df["on_IsoTF"] = (
    merged_master_df["TFIso_clone_ids"].notna()
)

requested_columns = [
    "on_IsoTF",
    "total_PPI",
    "#_of_PPIs",
    "%_of_PPIs"
]

original_columns = [
    column for column in master_df.columns
    if column not in requested_columns
]

merged_master_df = merged_master_df[
    original_columns + requested_columns
]

merged_master_df.to_csv(OUTPUT_PATH, index=False)

match_audit_df = master_with_keys[
    list(master_df.columns) + ["gene_key", "sequence_key"]
].merge(
    match_table,
    on=["gene_key", "sequence_key"],
    how="left",
    validate="many_to_one"
)

match_audit_df.to_csv(AUDIT_PATH, index=False)

print("Saved merged master:")
print(OUTPUT_PATH)

print("\nSaved sequence-match audit:")
print(AUDIT_PATH)

display(
    merged_master_df.loc[
        merged_master_df["on_IsoTF"],
        [
            MASTER_GENE_COL,
            "on_IsoTF",
            "total_PPI",
            "#_of_PPIs",
            "%_of_PPIs"
        ]
    ].head(20)
)


## 5. Coverage and sparsity statistics

In [ ]:

raw_stats = pd.Series({
    "TFIso library clones":
        library["clone_id"].nunique(),

    "TFIso library genes":
        library["gene_key"].nunique(),

    "TF isoforms with >=1 interpretable Y2H test":
        tested_ppi["ad_clone_id"].nunique(),

    "TF genes with >=1 interpretable Y2H test":
        tested_ppi["gene_key"].nunique(),

    "TF isoforms with >=1 positive PPI":
        positive_ppi["ad_clone_id"].nunique(),

    "TF genes with >=1 positive PPI":
        positive_ppi["gene_key"].nunique(),

    "Unique positive TF-isoform/partner PPIs":
        positive_ppi[
            ["ad_clone_id", "db_gene_symbol"]
        ].drop_duplicates().shape[0],

    "Unique gene-level PPI partners":
        positive_ppi[
            ["gene_key", "db_gene_symbol"]
        ].drop_duplicates().shape[0]
}, name="count")

master_stats = pd.Series({
    "Master isoform rows":
        len(merged_master_df),

    "Master TF genes":
        normalize_gene(
            merged_master_df[MASTER_GENE_COL]
        ).nunique(),

    "Master isoforms found in TFIso":
        int(merged_master_df["on_IsoTF"].sum()),

    "Master genes with >=1 isoform found in TFIso":
        normalize_gene(
            merged_master_df.loc[
                merged_master_df["on_IsoTF"],
                MASTER_GENE_COL
            ]
        ).nunique(),

    "Matched master isoforms with usable PPI data":
        int(merged_master_df["#_of_PPIs"].notna().sum()),

    "Matched master isoforms with >=1 positive PPI":
        int(merged_master_df["#_of_PPIs"].gt(0).sum()),

    "Matched master genes with usable PPI data":
        normalize_gene(
            merged_master_df.loc[
                merged_master_df["#_of_PPIs"].notna(),
                MASTER_GENE_COL
            ]
        ).nunique()
}, name="count")

print("Raw TFIso/Y2H coverage")
display(raw_stats.to_frame())

print("Coverage after exact sequence matching to the master")
display(master_stats.to_frame())

print("PPI count distribution among matched, tested master isoforms")
display(
    merged_master_df["#_of_PPIs"]
    .dropna()
    .describe()
    .to_frame("value")
)

print("Gene-level total PPI distribution")
display(
    gene_ppi_summary["total_PPI"]
    .describe()
    .to_frame("value")
)


## 6. Simple summaries

In [ ]:

top_isoforms = (
    merged_master_df.loc[
        merged_master_df["#_of_PPIs"].notna(),
        [
            MASTER_GENE_COL,
            "total_PPI",
            "#_of_PPIs",
            "%_of_PPIs"
        ]
    ]
    .sort_values(
        ["#_of_PPIs", "%_of_PPIs"],
        ascending=False
    )
)

print("Top matched TF isoforms by number of positive PPI partners")
display(top_isoforms.head(25))

plot_data = merged_master_df["#_of_PPIs"].dropna()

if len(plot_data) > 0:
    fig, ax = plt.subplots(figsize=(7, 5))
    bins = np.arange(
        -0.5,
        plot_data.max() + 1.5,
        1
    )
    ax.hist(plot_data, bins=bins)
    ax.set_xlabel("Number of positive Y2H PPI partners per isoform")
    ax.set_ylabel("Number of matched TF isoforms")
    ax.set_title("TFIso PPI counts among master-table isoforms")
    fig.tight_layout()
    plt.show()



## Output interpretation

- `on_IsoTF = False`: no exact gene + amino-acid sequence match was found in the TFIso library.
- `on_IsoTF = True`, PPI columns missing: the isoform is in TFIso but lacks usable pairwise Y2H data.
- `#_of_PPIs = 0`: the isoform had at least one interpretable Y2H test but no positive PPI.
- `total_PPI`: union of positive partners observed across all tested isoforms of that TF gene.
- `%_of_PPIs`: fraction of the gene's observed PPI repertoire recruited by that isoform.

The merged file is written to:

`TranscriptionIsoforms_df_geneage_TFIso_PPI.csv`
